In [1]:
import pandas as pd

In [2]:
#df = pd.read_csv('gold_ml_price_features.csv')
df = pd.read_csv('cz_real_estate.csv')

In [3]:
df.head(5)

,title,price,currency,location,description_en,description_native,construction_of_building,condition,equipped,area_of_property,...,age,garage,elevator,balcony,parking,barrier_free_access,cellar,near_public_transport,terrace,transaction_type
0,Inzerát již není v nabídce Pronájem bytu 1+1 •...,8000.0,CZK,"sídliště 9. května, Nejdek - Nejdek, Karlovars...",NaN,Rádi bychom Vám představili tento byt k pronáj...,Cihla,Dobrý,Částečně,NaN,...,NaN,0,0,0,0,0,1,0,0,rent
1,Inzerát již není v nabídce Prodej bytu 6+1 • 2...,534000.0,EUR,Bozener Str 64 Ludwigshafen Gartenstadt Rheinl...,NaN,"Na prodej je balík, který zahrnuje prostorný b...",NaN,NaN,NaN,NaN,...,Nad 50 let,1,1,1,0,1,1,1,0,sale
2,Inzerát již není v nabídce Prodej domu 7+1 • 2...,749000.0,EUR,", Severní Porýní-Vestfálsko",NaN,Níže naleznete překlad textu označeného XML ta...,NaN,Po rekonstrukci,NaN,850.0,...,Nad 50 let,1,0,1,1,0,1,1,1,sale
3,Inzerát již není v nabídce Prodej bytu 2+1 • 3...,165000.0,EUR,Mendelsohnstraße 12 Augsburg Oberhausen Bayern...,NaN,Atraktivní 2-pokojový investiční byt a byt pro...,NaN,Před rekonstrukcí,NaN,NaN,...,Nad 50 let,0,0,0,0,0,1,1,0,sale
4,Inzerát již není v nabídce Prodej bytu 4+1 • 9...,389000.0,EUR,"Augsburger Straße 29b, , Bavorsko",NaN,"Popis bytu Königsbrunn, Augsburger Straße\n• O...",NaN,Po rekonstrukci,NaN,NaN,...,Nad 50 let,1,1,1,0,0,1,1,0,sale


In [4]:
def create_textual_representation(row):
    textual_representation = f""" Title: {row["title"]},
Price: {row["price"]} {row["currency"]},

Description_in_native: {row["description_native"]},

Description_in_english: {row["description_en"]}""" # Let's test these features for now
    return textual_representation

In [5]:
df['textual_representation'] = df.apply(create_textual_representation, axis=1)

In [6]:
print(df['textual_representation'].values[0])

 Title: Inzerát již není v nabídce Pronájem bytu 1+1 • 37 m² bez realitky sídliště 9. května, Nejdek - Nejdek, Karlovarský kraj,
Price: 8000.0 CZK,

Description_in_native: Rádi bychom Vám představili tento byt k pronájmu v městě Nejdek. Tento byt o dispozici 1+1 se nachází v cihlové budově a je v osobním vlastnictví. Byt je umístěn v prvním patře a nabízí užitnou plochu 37 m2, což je ideální pro jednotlivce nebo pár.
Byt je v dobrém stavu a je připraven k okamžitému nastěhování od 1. února 2026. Disponuje také sklepem o velikosti 6 m2, který poskytuje dostatek úložného prostoru pro Vaše věci. Za byt se platí nájem ve výši 8 000 Kč měsíčně. Poplatek za služby se hradí zvlášť a činí přibližně 2 600 Kč při bydlení jedné osoby. Elektřina a plyn budou převedeny na nájemce.
Byt je částečně zařízený. Postel je rozkládací a nabízí spací plochu o rozměrech 160 × 200 cm. Z vybavení je v koupelně k dispozici pračka, v kuchyni lednice, trouba a varná deska.
Vratná kauce činí 14 000 Kč. Nájemní sml

In [7]:
import faiss
import requests
import numpy as np

dim = 4096 #emb dim

index = faiss.IndexFlatL2(dim)

X = np.zeros((len(df['textual_representation']), dim), dtype = 'float32')

In [13]:
for i, representation in enumerate(df['textual_representation']):
    print('Processed', str(i), '/', len(df), 'instances')

    res = requests.post('http://localhost:11434/api/embeddings',
                        json={
                            'model': 'llama3.1:8b',
                            'prompt': representation
                        }
                       )
    embedding = res.json()['embedding']

    X[i] = np.array(embedding)

index.add(X)

Processed 0 / 9337 instances
Processed 1 / 9337 instances
Processed 2 / 9337 instances
Processed 3 / 9337 instances
Processed 4 / 9337 instances
Processed 5 / 9337 instances
Processed 6 / 9337 instances
Processed 7 / 9337 instances
Processed 8 / 9337 instances
Processed 9 / 9337 instances
Processed 10 / 9337 instances
Processed 11 / 9337 instances
Processed 12 / 9337 instances
Processed 13 / 9337 instances
Processed 14 / 9337 instances
Processed 15 / 9337 instances
Processed 16 / 9337 instances
Processed 17 / 9337 instances
Processed 18 / 9337 instances
Processed 19 / 9337 instances
Processed 20 / 9337 instances
Processed 21 / 9337 instances


KeyboardInterrupt: 

In [14]:
X

array([[-0.21065532, -1.8849174 , -3.3718998 , ...,  0.2572542 ,
         1.4580442 , -1.9220397 ],
       [ 1.0871043 , -1.7796571 , -3.2599697 , ..., -1.3833858 ,
         2.1755376 , -2.556581  ],
       [ 0.12415119, -1.7804271 , -2.997074  , ..., -1.5545841 ,
         1.904199  , -2.4972107 ],
       ...,
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ]], shape=(9337, 4096), dtype=float32)

In [ ]:
faiss.write_index(index, 'index')

In [ ]:
index = faiss.inde_index('index')

## This is too slow.
Try to use dedicated embedding model, or batch with async requests.
Also save embeddings to disk, to not recompute them every run